In [2]:
import MLM.moire_lattice_vector_finder as mlf
import numpy as np
import pandas as pd
import time
import sys, os
from pprint import pprint

sys.path.append("../AtomLayers/refactor")

In [3]:
from match import run_and_filter

In [4]:
file_1 = "../moire_structures/bto/bto_5-layer.vasp"
file_2 = "../moire_structures/bto/bto_5-layer.vasp"

In [5]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

In [6]:
lattice_vectors1['a']

array([3.9903791, 0.       , 0.       ])

In [7]:
lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

In [8]:
mlf.angle_between(lattice_vectors1['a'],lattice_vectors1['b'])

90.0

In [11]:
results = run_and_filter(a1=lattice_vectors1['a'][0:2],a2=lattice_vectors1['b'][0:2],
                      g1=lattice_vectors2['a'][0:2],g2=lattice_vectors2['b'][0:2],
                      degmin=3.90, degmax = 4.00, degstep = 0.01,
                      dtol = 5e-4, nmin = -40, nmax = 40, goodangles = [90.0])

In [12]:
angle=[]
A1 = []
A2 = []
delvec = []
delgood = []
norm_A1 = []
norm_A2 = []
#num_atoms_app = []

for result in results[0]['results']:
    # pprint(len(result))
    # pprint(result)
    # pprint("#################")
    # pprint(result[0]['angle'])
    angle.append(result[0]['angle'])
    # pprint(result[0]['matchpoint'])
    A1.append(result[0]['matchpoint'])
    # pprint(result[1]['matchpoint'])
    A2.append(result[1]['matchpoint'])
    # pprint(result[2]['delgoodangle'])
    delvec.append(result[0]['delvec'])
    delgood.append(np.abs(result[2]['delgoodangle']))
    norm_A1.append(np.linalg.norm(np.array(result[0]['matchpoint'])))
    norm_A2.append(np.linalg.norm(np.array(result[1]['matchpoint'])))
    #num_atoms_app.append(np.abs(density * np.cross(np.array(result[0]['matchpoint']), np.array(result[1]['matchpoint']))))
    
    
    # break

In [13]:
resdf = pd.DataFrame(zip(angle,A1,A2,delvec,delgood,norm_A1, norm_A2), columns=['Theta','A1', 'A2', 'delvec', 'delang','norm_A1', 'norm_A2'])

In [14]:
len(resdf)

40

In [17]:
# Sort the DataFrame by 'degree' in ascending order and 'norm' in descending order, followed by 'vec_del' and 'delta_angle'
rsdf_sorted = resdf.sort_values(by=[ 'delvec'], ascending=[ True])



In [18]:
rsdf_sorted.head(60)

,Theta,A1,A2,delvec,delang,norm_A1,norm_A2
0,3.95,"[-59.8556864265, 55.865307331400004]","[-55.865307331400004, -59.8556864265]",0.000189,0.0,81.875734,81.875734
1,3.95,"[-59.8556864265, 55.865307331400004]","[55.865307331400004, 59.8556864265]",0.000189,0.0,81.875734,81.875734
2,3.95,"[-59.8556864265, 55.865307331400004]","[-111.73061466280001, -119.711372853]",0.000189,0.0,81.875734,163.751467
3,3.95,"[-59.8556864265, 55.865307331400004]","[111.73061466280001, 119.711372853]",0.000189,0.0,81.875734,163.751467
4,3.95,"[-55.865307331400004, -59.8556864265]","[-59.8556864265, 55.865307331400004]",0.000189,0.0,81.875734,81.875734
5,3.95,"[-55.865307331400004, -59.8556864265]","[59.8556864265, -55.865307331400004]",0.000189,0.0,81.875734,81.875734
6,3.95,"[-55.865307331400004, -59.8556864265]","[-119.711372853, 111.73061466280001]",0.000189,0.0,81.875734,163.751467
7,3.95,"[-55.865307331400004, -59.8556864265]","[119.711372853, -111.73061466280001]",0.000189,0.0,81.875734,163.751467
8,3.95,"[55.865307331400004, 59.8556864265]","[-59.8556864265, 55.865307331400004]",0.000189,0.0,81.875734,81.875734
9,3.95,"[55.865307331400004, 59.8556864265]","[59.8556864265, -55.865307331400004]",0.000189,0.0,81.875734,81.875734


In [19]:
# Group by 'degree' and take the first entry in each group
final_rsdf = rsdf_sorted.groupby('delvec', as_index=False).first()

In [20]:
len(final_rsdf)

3

In [21]:
# final_rsdf['num_atoms'] = np.round(1.574158*final_rsdf['norm_A1']*final_rsdf['norm_A2'])

In [22]:
final_rsdf.sort_values(by="delvec", ascending=True).head(60)

,delvec,Theta,A1,A2,delang,norm_A1,norm_A2
0,0.000189,3.95,"[-59.8556864265, 55.865307331400004]","[-55.865307331400004, -59.8556864265]",0.0,81.875734,81.875734
1,0.000267,3.95,"[-115.7209937579, -3.9903790951]","[-3.9903790951, 115.7209937579]",0.0,115.789773,115.789773
2,0.000377,3.95,"[-119.711372853, 111.73061466280001]","[-55.865307331400004, -59.8556864265]",0.0,163.751467,81.875734


In [36]:
tol = 0.05  # Theta‐difference tolerance for “closeness”

# 1) sort by Theta
df_sorted = final_rsdf.sort_values('Theta').reset_index(drop=True)

keep_idxs = []           # indices of the rows we'll keep
current_group = [0]      # start with the first row in group

for i in range(1, len(df_sorted)):
    θ_prev = df_sorted.loc[i-1, 'Theta']
    θ_curr = df_sorted.loc[i,   'Theta']

    if abs(θ_curr - θ_prev) <= tol:
        # same cluster: add to current_group
        current_group.append(i)
    else:
        # new cluster starts: pick best from the old
        best = df_sorted.loc[current_group, 'delvec'].idxmin()
        keep_idxs.append(best)
        current_group = [i]

# final cluster
best = df_sorted.loc[current_group, 'delvec'].idxmin()
keep_idxs.append(best)

# select and restore original order (optional)
result = df_sorted.loc[keep_idxs].sort_index().reset_index(drop=True)

In [41]:
result.sort_values(by="Theta", ascending=True).head(60)

,Theta,A1,A2,delvec,delang,norm_A1,norm_A2
0,2.19,"[-47.884549141200004, -67.8364446167]","[-67.8364446167, 47.884549141200004]",0.008979,0.0,83.034410,83.034410
1,5.91,"[-35.9134118559, 7.9807581902]","[-7.9807581902, -35.9134118559]",0.015384,0.0,36.789477,36.789477
2,7.17,"[-55.865307331400004, 39.903790951000005]","[-39.903790951000005, -55.865307331400004]",0.016923,0.0,68.653078,68.653078
3,8.73,"[-75.8172028069, 7.9807581902]","[-7.9807581902, -75.8172028069]",0.016164,0.0,76.236086,76.236086
4,9.52,"[-75.8172028069, -63.8460655216]","[-63.8460655216, 75.8172028069]",0.018810,0.0,99.118960,99.118960
5,10.46,"[-63.8460655216, 3.9903790951]","[-3.9903790951, -63.8460655216]",0.012566,0.0,63.970643,63.970643
6,11.42,"[-79.80758190200001, -35.9134118559]","[35.9134118559, -79.80758190200001]",0.012068,0.0,87.515846,87.515846
7,11.75,"[-55.865307331400004, 39.903790951000005]","[-39.903790951000005, -55.865307331400004]",0.017162,0.0,68.653078,68.653078
8,15.22,"[-43.8941700461, -15.9615163804]","[-15.9615163804, 43.8941700461]",0.009811,0.0,46.706190,46.706190
9,17.38,"[-67.8364446167, 47.884549141200004]","[-47.884549141200004, -67.8364446167]",0.009402,0.0,83.034410,83.034410


In [23]:
final_rsdf.to_pickle("../moire_structures/bto/bto_bilayer_5l.pkl")

In [39]:
len(result)

32